In [1]:
# # Q-Learning 逃離 21×11 迷宮（含 5 個寶藏）
# - 最多訓練 1000 回合
# - 每回合限 300 步
# - 收集 5 個寶藏後到達終點才算成功
# - 獎勵設計：終點 +50，寶藏 +10（每回合每格僅得一次），撞牆 –10，其他步 –1
import numpy as np
import random

In [10]:
# ─── 一、參數與地圖設定 ────────────────────────────────────
WIDTH, HEIGHT = 21, 11
NUM_STATES = WIDTH * HEIGHT
ACTIONS = ['up','down','left','right']

START = (0, 0)
GOAL  = (20, 10)
MAX_EPISODES = 1000
MAX_STEPS    = 1000

EPSILON     = 0.9
EPS_DECAY   = 0.995
EPS_MIN     = 0.01
ALPHA       = 0.1
GAMMA       = 0.9

# **請把下面兩行替換成作業提供的牆壁與寶藏座標列表**
walls_coords = [
    (0,4),(0,5),(0,7),(0,9),(1,1),(1,2),(1,4),(1,9),(1,10),(1,14),(1,18),(2,1),(2,3),(2,5),(2,7),(2,8),(2,9),(2,11),(2,13),(2,15),(2,16),(2,17),(2,19),(3,2),(3,8),(3,11),(3,17),(4,1),(4,4),(4,6),(4,10),(4,13),(4,16),(4,17),(4,18),(4,20),(5,4),(5,5),(5,6),(5,8),(5,9),(5,14),(5,15),(6,1),(6,2),(6,3),(6,6),(6,8),(6,10),(6,15),(6,16),(6,17),(6,19),(7,4),(7,6),(7,8),(7,10),(7,11),(7,17),(7,19),(8,1),(8,4),(8,8),(8,10),(8,13),(8,15),(8,18),(8,19),(9,1),(9,2),(9,4),(9,6),(9,7),(9,17),(10,1),(10,4),(10,16),(10,19)
]
treasure_coords = [
    (0, 6),(3, 16),(8, 2),(10, 2),(10, 17)
]

walls     = set(walls_coords)
treasures = set(treasure_coords)

In [11]:
# 轉換工具：二維座標 ↔ 一維索引
def to_index(pos):
    x,y = pos
    return y * WIDTH + x

def to_pos(idx):
    return (idx % WIDTH, idx // WIDTH)

# 移動函式：撞牆則留在原地
def move(pos, action):
    x,y = pos
    if   action=='up':    y = max(0,        y-1)
    elif action=='down':  y = min(HEIGHT-1, y+1)
    elif action=='left':  x = max(0,        x-1)
    elif action=='right': x = min(WIDTH-1,  x+1)
    new = (x,y)
    return pos if new in walls else new

# ## 二、初始化 Q-table
Q = np.zeros((NUM_STATES, len(ACTIONS)))

best_path  = None
best_steps = MAX_STEPS + 1
best_score = -1

epsilon = EPSILON

In [ ]:
# ## 三、Q-Learning 訓練迴圈
for episode in range(1, MAX_EPISODES+1):
    pos      = START
    state    = to_index(pos)
    collected = set()
    path      = [state]
    score     = 0

    for step in range(1, MAX_STEPS+1):
        # 1) ε-貪婪選動作
        if random.random() < epsilon:
            a = random.choice(ACTIONS)
        else:
            a = ACTIONS[np.argmax(Q[state])]

        # 2) 執行動作，得到下一狀態、獎勵、是否結束
        new_pos   = move(pos, a)
        new_state = to_index(new_pos)

        # 獎勵設計
        if new_pos == pos and new_pos in walls:
            r = -10                # 撞牆
        elif new_pos in treasures and new_pos not in collected:
            r = +10                # 第一次拿到寶藏
            collected.add(new_pos)
            score += 1
        elif new_pos == GOAL:
            r = +50                # 到達終點
        else:
            r = -1                 # 一般步數懲罰

        # 3) Q 值更新
        idx = ACTIONS.index(a)
        q_predict = Q[state, idx]
        if new_pos == GOAL:
            q_target = r
        else:
            q_target = r + GAMMA * np.max(Q[new_state])
        Q[state, idx] += ALPHA * (q_target - q_predict)

        # 4) 更新狀態與記錄路徑
        pos   = new_pos
        state = new_state
        path.append(state)

        # 5) 如果到終點就結束回合
        if new_pos == GOAL:
            break

    # ε 衰減
    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)

    # 記錄最佳（僅考慮收集全部寶藏並到終點的回合）
    if pos == GOAL and score == len(treasure_coords):
        if step < best_steps:
            best_steps = step
            best_score = score
            best_path  = path.copy()

    # （可選）每 100 回合印一次進度
    if episode % 100 == 0:
        print(f'Ep{episode:4d} | ε={epsilon:.3f} | best_steps={best_steps}')

# ## 四、輸出並儲存結果
if best_path is None:
    print("警告：訓練結束，未找到同時收集 5 顆寶藏且到終點的路徑")
else:
    print("=== 最佳結果 ===")
    print(f"步數：{best_steps}，寶藏：{best_score}（應為 {len(treasure_coords)}）")

    # 儲存 Q-table
    np.save('q_table.npy', Q)
    print("已儲存 Q-table → q_table.npy")

    # 文字化路徑顯示
    maze = [[' ']*WIDTH for _ in range(HEIGHT)]
    for x,y in walls:     maze[y][x] = 'X'
    for x,y in treasures: maze[y][x] = 'O'
    sx,sy = START; gx,gy = GOAL
    maze[sy][sx] = 'S'; maze[gy][gx] = 'G'
    for idx in best_path:
        x,y = to_pos(idx)
        if maze[y][x] == ' ':
            maze[y][x] = '.'

    print("\n路徑（. 為走過的路徑）：")
    for row in maze:
        print(''.join(row))

Ep 100 | ε=0.545 | best_steps=1001
Ep 200 | ε=0.330 | best_steps=1001
Ep 300 | ε=0.200 | best_steps=1001
Ep 400 | ε=0.121 | best_steps=1001
Ep 500 | ε=0.073 | best_steps=1001
Ep 600 | ε=0.044 | best_steps=1001
Ep 700 | ε=0.027 | best_steps=1001
Ep 800 | ε=0.016 | best_steps=1001
Ep 900 | ε=0.010 | best_steps=1001
Ep1000 | ε=0.010 | best_steps=1001
